# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

### Службова клітинка: таймер виконання

Ця клітинка потрібна для відповіді на питання про тривалість виконання: вона підключає до IPython обробник, який після кожної клітинки записує, скільки секунд вона виконувалася. Запускайте її **першою**.

In [1]:
# Службова клітинка до питання про тривалість: запамʼятовує, скільки секунд виконувалася кожна клітинка.
# Запускайте її першою, до всіх інших клітинок ноутбука.
import json
import os
import time

from IPython import get_ipython

TIMES_FILE = "cell_times_03.json"  # файл переживе перезапуск середовища в межах сесії Colab
CELL_TIMES = json.load(open(TIMES_FILE)) if os.path.exists(TIMES_FILE) else []
_started = [None]


def _pre_run(*args):
    _started[0] = time.perf_counter()


def _post_run(result=None):
    if _started[0] is None:  # саму службову клітинку не вимірюємо
        return
    raw = (getattr(getattr(result, "info", None), "raw_cell", "") or "").strip()
    CELL_TIMES.append({
        "cell": getattr(result, "execution_count", None),
        "sec": round(time.perf_counter() - _started[0], 1),
        "code": raw.splitlines()[0][:70] if raw else "",
    })
    with open(TIMES_FILE, "w") as f:
        json.dump(CELL_TIMES, f)


ip = get_ipython()
for event, callback in (("pre_run_cell", _pre_run), ("post_run_cell", _post_run)):
    for old in list(ip.events.callbacks[event]):
        if getattr(old, "__name__", "") == callback.__name__:
            ip.events.unregister(event, old)
    ip.events.register(event, callback)
print("Таймер клітинок увімкнено")

Таймер клітинок увімкнено


In [2]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_464/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [4]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [5]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [7]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [8]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [9]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [10]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [11]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [12]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [13]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [14]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [15]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [16]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

## Відповіді на теоретичні питання (частина 0.1, питання 21–30)

Відповіді підтверджено кодом, який використовує змінні попередніх розділів (`docs`, `chunks`, `vector_store`, `bm25`, `hybrid_search`, `reranker`, `BENCHMARK`, `evaluation` тощо), тому секцію слід запускати **після** всіх клітинок вище і **по порядку**: деякі клітинки використовують результати попередніх (наприклад, `chunk_matrix` з питання 23 використовується в питаннях 25 і 26).

### Питання 21. Як довго виконувалися клітинки ноутбуку 03_rag_langchain_eval? Чи були проблеми з запуском, які?

In [17]:
import pandas as pd


def timing_report(top=6):
    """Скільки часу забрали клітинки ноутбука (за даними службової клітинки з початку ноутбука)."""
    times = pd.DataFrame(CELL_TIMES)
    print(f"Виконано клітинок: {len(times)}; сумарний час їх виконання: {times['sec'].sum() / 60:.1f} хв")
    print(f"Найдовші {top} клітинок:")
    return times.sort_values("sec", ascending=False).head(top)[["cell", "sec", "code"]].reset_index(drop=True)


timing_report()

Виконано клітинок: 15; сумарний час їх виконання: 9.8 хв
Найдовші 6 клітинок:


,cell,sec,code
0,13,251.2,"GENERATOR_MODEL_NAME = ""Qwen/Qwen3-4B-Instruct..."
1,15,116.8,"REFUSAL = ""немає відповіді"""
2,4,57.2,# @title Завантаження й очищення документів (я...
3,3,51.6,import re
4,10,38.7,"reranker = CrossEncoder(""BAAI/bge-reranker-v2-..."
5,7,36.4,"EMBEDDING_MODEL_NAME = ""Qwen/Qwen3-Embedding-0..."


**Тривалість.** Службова клітинка на початку ноутбука виміряла 15 клітинок, виконаних до цього місця; разом вони працювали **9,8 хв**. Шість найдовших:

- завантаження генеративної моделі Qwen3-4B-Instruct — 251,2 с (понад 4 хв, близько 43 % усього часу);
- оцінювання всього бенчмарку (26 питань: відповідь ланцюжка й вердикт судді) — 116,8 с, тобто приблизно 4,5 с на питання;
- завантаження й очищення документів — 57,2 с;
- імпорти — 51,6 с;
- завантаження cross-encoder — 38,7 с;
- завантаження моделі ембедингів — 36,4 с.

Ці шість клітинок дають близько 94 % часу. Отже, час забирають одноразові завантаження моделей (генератор, cross-encoder і ембединги разом — 326 с, понад половина), а не робота на запит. Клітинки відповідей на питання 22–30 виконувалися вже після цього виводу й у суму не входять; загального виміру для них немає, окремо виміряно лише переранжування всього корпусу в питанні 26 (≈ 4,8 с на питання).

**Проблеми з запуском.** Жодна клітинка не впала й не перезапускалась: лічильники виконання йдуть підряд від 1 до 27. Були три попередження, що на результати не впливають:

- `DeprecationWarning` від `langchain-community` у клітинці з імпортами: пакет більше не підтримується активно, а `BM25Retriever` імпортується саме з нього, тому в майбутніх версіях цей імпорт може зникнути; для відтворюваності варто зафіксувати версії пакетів;
- нагадування Hugging Face Hub, що запити надсилаються без токена `HF_TOKEN` (нижчі ліміти завантаження);
- повідомлення `transformers`, що конвеєр на GPU викликається послідовно: питання оцінюються по одному, а не пакетами. Це лише втрата швидкості.

### Питання 22. Чим RecursiveCharacterTextSplitter (розділ 2) відрізняється від розбиття за реченнями з демо 2? Для чого задано параметр keep_separator="end"?

**Відмінності.** Розбиття за реченнями з демо 2 спершу ділить текст на речення регулярним виразом (який не вважає кінцем речення номери пунктів на зразок «1.4.» і ріже надто довгі «речення»-таблиці за довжиною), а потім пакує **цілі речення** у фрагмент, доки він не перевищить `CHUNK_SIZE`; перекриттям слугує останнє речення. `RecursiveCharacterTextSplitter` не знає, що таке речення: він пробує розділювачі за списком пріоритетів (`". "`, `"; "`, `" "`, `""`), рекурсивно переходить до дрібнішого розділювача лише для шматків, що не вміщуються у `chunk_size`, і збирає з шматків фрагменти, а перекриття складає зі шматків загальною довжиною до `chunk_overlap` символів. Розділювач `". "` він шукає як звичайний рядок, тому «1.4. » теж вважає межею.

**Навіщо `keep_separator="end"`.** За замовчуванням розділювач при розбитті **відкидається**. Тоді фрагменти втрачали б крапки й крапки з комою, що в кінці речення, і текст у фрагменті виглядав би обірваним. Значення `"end"` залишає розділювач **в кінці попереднього шматка**, тож фрагмент закінчується на «.» чи «;», як і належить реченню (значення `"start"` перенесло б розділювач на початок наступного фрагмента, і той починався б з «. »). Код нижче порівнює обидва підходи за вимірюваними показниками й показує дію `keep_separator` на короткому тексті.

In [18]:
SENTENCE_END = re.compile(r"(?<=[.!?])(?<!\d\.)\s+(?=[А-ЯІЇЄҐA-Z«\d])")  # з демо 2


def split_sentences(text, size=1000):
    sentences = []
    for sentence in SENTENCE_END.split(text):
        sentences += [sentence[i:i + size] for i in range(0, len(sentence), size)]
    return sentences


def pack_sentences(sentences, size=1000):
    packed, current = [], []
    for sentence in sentences:
        if current and len(" ".join(current + [sentence])) > size:
            packed.append(" ".join(current))
            current = current[-1:] if len(current[-1]) + len(sentence) < size else []
        current.append(sentence)
    packed.append(" ".join(current))
    return packed


# Фрагменти без дописаної назви документа: рівно те, що повертають розбивачі
recursive_raw = [(d.metadata["doc_id"], d.page_content) for d in splitter.split_documents(docs)]
sentence_raw = [(doc.metadata["doc_id"], text) for doc in docs for text in pack_sentences(split_sentences(doc.page_content))]


def mean_overlap(pairs):
    """Середня довжина (у символах) спільної частини сусідніх фрагментів одного документа."""
    overlaps = []
    for (doc_a, a), (doc_b, b) in zip(pairs, pairs[1:]):
        if doc_a == doc_b:
            overlaps.append(next((n for n in range(min(len(a), len(b), 400), 0, -1) if a.endswith(b[:n])), 0))
    return float(np.mean(overlaps))


def chunk_stats(name, pairs):
    texts = [t for _, t in pairs]
    return {
        "розбиття": name,
        "фрагментів": len(texts),
        "середня довжина": np.mean([len(t) for t in texts]),
        "макс. довжина": max(len(t) for t in texts),
        "починається зі строчної": np.mean([bool(re.match(r"[а-яіїєґa-z]", t.lstrip())) for t in texts]),
        "закінчується на . ! ? ; :": np.mean([bool(re.search(r"[.!?;:»)]\s*$", t)) for t in texts]),
        "закінчується номером пункту": np.mean([bool(re.search(r"(?:^|\s)\d+(?:\.\d+)*\.\s*$", t)) for t in texts]),
        "середнє перекриття, символів": mean_overlap(pairs),
    }


display(pd.DataFrame([chunk_stats("RecursiveCharacterTextSplitter", recursive_raw), chunk_stats("за реченнями (демо 2)", sentence_raw)]).round(2))

for name, pairs in [("RecursiveCharacterTextSplitter", recursive_raw), ("за реченнями (демо 2)", sentence_raw)]:
    sample = [t for doc_id, t in pairs if doc_id == "336"][:2]
    print(f"\n{name}, документ 336: кінець фрагмента 0 і початок фрагмента 1")
    print(f"   …{sample[0][-110:]}»")
    print(f"   «{sample[1][:110]}…")

print("\nДія keep_separator на короткому тексті (chunk_size=60, без перекриття):")
sample_text = "Студент вносить авансовий платіж до 1 вересня. Договір укладається на один рік. Гості можуть бути з 8:00 до 22:00."
for mode in ["end", "start", False]:
    parts = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0, separators=[". ", " ", ""], keep_separator=mode).split_text(sample_text)
    print(f"  keep_separator={mode!r:7}: {parts}")

,розбиття,фрагментів,середня довжина,макс. довжина,починається зі строчної,закінчується на . ! ? ; :,закінчується номером пункту,"середнє перекриття, символів"
0,RecursiveCharacterTextSplitter,385,792.06,999,0.05,0.93,0.44,59.49
1,за реченнями (демо 2),410,846.91,1000,0.09,0.86,0.00,125.55



RecursiveCharacterTextSplitter, документ 336: кінець фрагмента 0 і початок фрагмента 1
   …або за їх дорученням. У своїй роботі староста академічної групи співпрацює з куратором та ментором групи. 1.3.»
   «У своїй роботі староста академічної групи співпрацює з куратором та ментором групи. 1.3. У виборах старости ст…

за реченнями (демо 2), документ 336: кінець фрагмента 0 і початок фрагмента 1
   …атом або за їх дорученням. У своїй роботі староста академічної групи співпрацює з куратором та ментором групи.»
   «У своїй роботі староста академічної групи співпрацює з куратором та ментором групи. 1.3. У виборах старости ст…

Дія keep_separator на короткому тексті (chunk_size=60, без перекриття):
  keep_separator='end'  : ['Студент вносить авансовий платіж до 1 вересня.', 'Договір укладається на один рік.', 'Гості можуть бути з 8:00 до 22:00.']
  keep_separator='start': ['Студент вносить авансовий платіж до 1 вересня', '. Договір укладається на один рік', '. Гості можуть бути з 8:00 до 2

**Коментар до чисел.** `RecursiveCharacterTextSplitter` дав **385** фрагментів, розбиття за реченнями — **410**; середня довжина 792 і 847 символів, максимум 999 і 1000 (обидва вкладаються в ліміт). Середнє перекриття сусідніх фрагментів — 59 символів у рекурсивного й 126 у розбиття за реченнями (там перекриттям слугує останнє речення).

Найцікавіше — закінчення фрагментів.

1. **Номер пункту наприкінці.** Крапкою або іншим розділовим знаком закінчується 93 % рекурсивних фрагментів проти 86 % за реченнями, і це виглядає як перевага. Але **44 %** рекурсивних фрагментів закінчуються *номером наступного пункту* («… співпрацює з куратором та ментором групи. 1.3.», приклад документа 336 у виводі), тоді як у розбиття за реченнями таких закінчень **0 %**. Без «висячих» номерів охайних закінчень у рекурсивного розбивача лишається близько 49 % (93 % − 44 %) проти 86 %. Причина зрозуміла з коду: розділювач `". "` ріже текст і після «1.3.», а коротенький шматок «1.3.» ще вміщується у фрагмент, коли наступне речення вже ні, тож саме він стає останнім. Регулярний вираз із демо 2 не вважає «1.3.» кінцем речення (`(?<!\d\.)`), тому так не робить. Зміст при цьому не втрачається: наступний фрагмент починається з того самого речення разом із номером («1.3. У виборах старости…»), але як окрема цитата фрагмент закінчується обірвано.
2. **Початок із малої літери** (початок із середини речення) рідкісний в обох: 5 % і 9 %. У розбиття за реченнями він трапляється частіше, бо «речення»-таблиці довші за 1000 символів і ріжуться за довжиною.

Про `keep_separator`: на короткому прикладі видно, що з `"end"` фрагменти закінчуються крапкою («…до 1 вересня.»), з `"start"` крапка переходить на початок наступного («. Договір укладається…»), а з `False` зникає взагалі («…до 1 вересня»). Тож `"end"` справді зберігає закінчення речення, але, як показують 44 %, не гарантує, що фрагмент закінчується змістовним реченням. На бенчмарку це пошуку не завадило (hit@10 векторного пошуку дорівнює 1,00, див. питання 28).

### Питання 23. Чому у розділі 3 для FAISS обрано DistanceStrategy.MAX_INNER_PRODUCT? Чи можна було б так зробити, якби ембединги не були нормалізовані?

`MAX_INNER_PRODUCT` змушує LangChain створити в FAISS індекс `IndexFlatIP`, який рахує **скалярний добуток** (чим більше, тим ближче). Ембединги нормалізовано (`normalize_embeddings=True`), тобто довжина кожного вектора дорівнює 1, а для таких векторів:

- скалярний добуток дорівнює косинусній подібності: $a \cdot b = \|a\|\|b\|\cos\theta = \cos\theta$;
- квадрат евклідової відстані повністю визначається скалярним добутком: $\|a-b\|^2 = 2 - 2\,a\cdot b$.

Отже, порядок знайдених фрагментів за скалярним добутком, за косинусом і за (зворотною) евклідовою відстанню **однаковий**. Скалярний добуток — найдешевша операція (одне множення матриці), а `EUCLIDEAN_DISTANCE` (типовий вибір LangChain) тут нічого б не змінив у порядку, лише дав би «відстані» замість «подібностей».

**Якби ембединги не були нормалізованими**, скалярний добуток дорівнював би $\|a\|\|b\|\cos\theta$: фрагменти з довшими векторами отримували б вищу оцінку незалежно від змісту, і порядок відрізнявся б від косинусного. Довжина запиту на порядок не впливає (вона масштабує всі оцінки однаково), а от довжини **документів** впливають. Вихід — нормалізувати вектори перед індексацією (у LangChain для цього є параметр `normalize_L2=True`, або `faiss.normalize_L2`): тоді скалярний добуток знову стає косинусом. Код нижче перевіряє це на справжніх ембедингах корпусу і на змодельованих ненормалізованих.

In [19]:
chunk_matrix = vector_store.index.reconstruct_n(0, vector_store.index.ntotal).astype(np.float32)
norms = np.linalg.norm(chunk_matrix, axis=1)
print(f"Тип індексу FAISS: {type(vector_store.index).__name__}; ембедингів: {chunk_matrix.shape[0]}, розмірність {chunk_matrix.shape[1]}")
print(f"Довжини векторів фрагментів: min {norms.min():.4f}, max {norms.max():.4f} (відхилення від 1 — похибка половинної точності)")

probe_questions = [question for question, _, _ in answerable][:12]
Q = np.array([embeddings.embed_query(question) for question in probe_questions], dtype=np.float32)
ip = Q @ chunk_matrix.T                                                    # скалярний добуток
cos = (Q / np.linalg.norm(Q, axis=1, keepdims=True)) @ (chunk_matrix / norms[:, None]).T
l2 = np.stack([((chunk_matrix - Q[i]) ** 2).sum(1) for i in range(len(Q))])   # квадрат евклідової відстані, прямо за визначенням

q_norms = np.linalg.norm(Q, axis=1)
print(f"\nДовжини векторів запитів: {q_norms.min():.4f}–{q_norms.max():.4f}")
print(f"max |‖a−b‖² − (2 − 2·a·b)| = {np.abs(l2 - (2 - 2 * ip)).max():.4f}  (для довжин ≈ 1 тотожність виконується з похибкою нормалізації)")


def top_sets(scores, k, descending=True):
    return [set((np.argsort(-s) if descending else np.argsort(s))[:k]) for s in scores]


def share_common(a, b, k):
    return float(np.mean([len(x & y) / k for x, y in zip(a, b)]))


ip_top1, cos_top1, l2_top1 = (top_sets(ip, 1), top_sets(cos, 1), top_sets(l2, 1, descending=False))
print(f"Найближчий фрагмент однаковий: скалярний добуток і косинус — {sum(x == y for x, y in zip(ip_top1, cos_top1))} з {len(Q)} запитів; "
      f"скалярний добуток і евклідова відстань — {sum(x == y for x, y in zip(ip_top1, l2_top1))} з {len(Q)}")
print(f"Частка спільних фрагментів у топ-10: скалярний добуток і косинус — {share_common(top_sets(ip, 10), top_sets(cos, 10), 10):.0%}; "
      f"скалярний добуток і евклідова відстань — {share_common(top_sets(ip, 10), top_sets(l2, 10, descending=False), 10):.0%}")

# Моделюємо ненормалізовані ембединги: кожен фрагмент множимо на випадковий коефіцієнт від 0.5 до 3
rng = np.random.default_rng(0)
scaled = chunk_matrix * rng.uniform(0.5, 3.0, size=(len(chunk_matrix), 1)).astype(np.float32)
ip_scaled = Q @ scaled.T
top1_changed = sum(int(np.argmax(ip_scaled[i]) != np.argmax(cos[i])) for i in range(len(Q)))
overlap5 = np.mean([len(set(np.argsort(-ip_scaled[i])[:5]) & set(np.argsort(-cos[i])[:5])) / 5 for i in range(len(Q))])
print(f"\nНенормалізовані (змодельовано): найближчий фрагмент змінився у {top1_changed} з {len(Q)} запитів; "
      f"середня частка спільних фрагментів у топ-5 з косинусним порядком: {overlap5:.0%}")

Тип індексу FAISS: IndexFlatIP; ембедингів: 385, розмірність 1024
Довжини векторів фрагментів: min 0.9998, max 1.0005 (відхилення від 1 — похибка половинної точності)

Довжини векторів запитів: 0.9998–1.0003
max |‖a−b‖² − (2 − 2·a·b)| = 0.0016  (для довжин ≈ 1 тотожність виконується з похибкою нормалізації)
Найближчий фрагмент однаковий: скалярний добуток і косинус — 12 з 12 запитів; скалярний добуток і евклідова відстань — 12 з 12
Частка спільних фрагментів у топ-10: скалярний добуток і косинус — 99%; скалярний добуток і евклідова відстань — 99%

Ненормалізовані (змодельовано): найближчий фрагмент змінився у 10 з 12 запитів; середня частка спільних фрагментів у топ-5 з косинусним порядком: 33%


**Коментар до результатів.** Індекс справді `IndexFlatIP` (385 фрагментів, розмірність 1024). Довжини векторів фрагментів — від 0,9998 до 1,0005, запитів — від 0,9998 до 1,0003, тобто нормалізація виконується (відхилення від 1 — похибка половинної точності). Тому:

- тотожність $\|a-b\|^2 = 2 - 2\,a\cdot b$ виконується з похибкою не більше 0,0016;
- найближчий фрагмент для всіх 12 перевірених запитів однаковий за скалярним добутком, косинусом і евклідовою відстанню (12 із 12); топ-10 збігається на 99 %, а не на 100 %. Розбіжність — приблизно одна позиція зі 120; найімовірніше, це майже рівні оцінки на межі топ-10, які похибка нормалізації переставляє.

Змодельовані ненормалізовані вектори (кожен фрагмент помножено на випадковий коефіцієнт від 0,5 до 3) картину ламають: найближчий фрагмент змінився у **10 із 12** запитів, а топ-5 збігається з косинусним порядком лише на **33 %**. Коефіцієнти взято довільно, щоб показати механізм, а не оцінити реальну модель: у справжніх ненормалізованих ембедингів розкид довжин буде іншим, але напрямок ефекту той самий. Отже, `MAX_INNER_PRODUCT` безпечний саме тому, що ембединги нормалізовані; для ненормалізованих потрібно спершу нормалізувати вектори (`normalize_L2=True`) або взяти косинусну подібність.

### Питання 24. Знайдіть власне питання до документів, на якому BM25 знаходить правильний фрагмент вище, ніж семантичний пошук, і питання, на якому навпаки (розділ 4). Поясніть, чому так відбувається.

BM25 порівнює **слова** (у нашому токенізаторі — точні словоформи в нижньому регістрі), тому вигравати має там, де запит дослівно містить рідкісні слова, коди й числа, що є у фрагменті: «F2», «FX», «за медичними показаннями». Він не знає, що «кіт», «собака» і «тварини» близькі за змістом, і не зводить словоформи до основи («самоплагіат» і «самоплагіатом» для нього різні слова). Семантичний пошук, навпаки, порівнює **зміст**: він знаходить фрагмент, що перефразовує питання («голова групи» і «староста», «виселитися» і «звільнити гуртожиток»), але може розмити точні рідкісні позначення, які для моделі мало відрізняються від сусідніх.

Замість того щоб шукати приклади навмання, перевіряємо чотирнадцять власних питань (частина написана майже дослівно за документами, частина — перефразована) і для кожного порівнюємо позицію правильного фрагмента в обох пошуках.

In [20]:
OWN_QUESTIONS = [
    ("Чи дозволено мешканцям гуртожитку мати кота чи собаку?", ["1219", "348"], r"утримувати тварин"),
    ("Кому призначають стипендію від ректора?", ["344"], r"встановлюється для студентів денної"),
    ("Через який час після початку навчання першокурсники вибирають голову групи?", ["336"], r"після двох тижнів"),
    ("Скільки днів дають на виселення, якщо студента відрахували?", ["1219"], r"протягом двох тижнів"),
    ("Як довго діє договір на проживання в гуртожитку?", ["348"], r"терміном на один навчальний рік"),
    ("Скільки триває каденція студентського омбудсмана?", ["4389"], r"терміном на 1 рік"),
    ("Який бал дає оцінку FX?", ["275"], r"35 ?– ?59"),
    ("Під яким кодом F2 записано спеціальність у чинній програмі?", ["3320"], r"спеціальн\w* F2"),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –"),
    ("Що розуміють під самоплагіатом?", ["1199"], r"Самоплагіат –"),
    ("Що таке фабрикація?", ["1199"], r"Фабрикація –"),
    ("Чи виплачують академічну стипендію щомісяця?", ["340"], r"один раз на місяць"),
    ("Яка допомога студентові в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної"),
    ("Скільки відсотків кредитів мають становити вибіркові курси?", ["277"], r"25 відсотків"),
]
MISSING = 99  # «не знайдено серед перших 20»

for question, gold, evidence in OWN_QUESTIONS:
    if not any(c.metadata["doc_id"] in gold and re.search(evidence, c.page_content) for c in chunks):
        print(f"УВАГА: вираз-доказ «{evidence}» не знайдено в жодному фрагменті — питання «{question}» не можна оцінити")

rows = []
for question, gold, evidence in OWN_QUESTIONS:
    rank_vector = first_hit_rank(vector_search(question, CANDIDATES), gold, evidence) or MISSING
    rank_bm25 = first_hit_rank(bm25_search(question, CANDIDATES), gold, evidence) or MISSING
    rows.append({"питання": question, "вектор": rank_vector, "BM25": rank_bm25,
                 "вище": "BM25" if rank_bm25 < rank_vector else "вектор" if rank_vector < rank_bm25 else "порівну"})
comparison = pd.DataFrame(rows)
display(comparison.replace({MISSING: ">20"}))
print("Позначка «>20» — правильного фрагмента немає серед 20 знайдених.\n")

comparison["перевага BM25"] = comparison["вектор"] - comparison["BM25"]
best_bm25 = comparison.loc[comparison["перевага BM25"].idxmax()]
best_vector = comparison.loc[comparison["перевага BM25"].idxmin()]
for title, row, sign in [("BM25 вище за вектор", best_bm25, 1), ("вектор вище за BM25", best_vector, -1)]:
    if sign * row["перевага BM25"] <= 0:
        print(f"{title}: таких питань серед перевірених немає\n")
        continue
    print(f"=== {title}: «{row['питання']}»  (позиція: вектор {row['вектор']}, BM25 {row['BM25']})")
    show(vector_search, row["питання"])
    show(bm25_search, row["питання"])

,питання,вектор,BM25,вище
0,Чи дозволено мешканцям гуртожитку мати кота чи...,1,2,вектор
1,Кому призначають стипендію від ректора?,2,1,BM25
2,Через який час після початку навчання першокур...,1,1,порівну
3,"Скільки днів дають на виселення, якщо студента...",2,>20,вектор
4,Як довго діє договір на проживання в гуртожитку?,1,1,порівну
5,Скільки триває каденція студентського омбудсмана?,1,4,вектор
6,Який бал дає оцінку FX?,4,4,порівну
7,Під яким кодом F2 записано спеціальність у чин...,5,>20,вектор
8,Що таке самоплагіат?,1,2,вектор
9,Що розуміють під самоплагіатом?,1,>20,вектор


Позначка «>20» — правильного фрагмента немає серед 20 знайдених.

=== BM25 вище за вектор: «Яка допомога студентові в академічній відпустці за медичними показаннями?»  (позиція: вектор 7, BM25 1)
vector_search: Яка допомога студентові в академічній відпустці за медичними показаннями?
  [340] Правила призначення академічних стипендій. У період тимчасової непрацездатності, підтвердженої довідкою заклад…
  [340] Правила призначення академічних стипендій. 3.10. Перевищення лімітів стипендіатів припускається в разі признач…
  [340] Правила призначення академічних стипендій. Студентам Житомирської політехніки з числа дітей- сиріт та дітей, п…

bm25_search: Яка допомога студентові в академічній відпустці за медичними показаннями?
  [340] Правила призначення академічних стипендій. Студентам, які відповідно до наказу ректора Житомирської політехнік…
  [340] Правила призначення академічних стипендій. Студентам Житомирської політехніки з числа дітей- сиріт та дітей, п…
  [1199] Кодекс академічної

**Приклади й пояснення за результатами.** З 14 питань вектор вище за BM25 у **8**, BM25 вище у **3**, порівну — у **3**. Вектор знайшов правильний фрагмент для всіх 14 питань (найгірша позиція — 7), BM25 не знайшов його серед 20 кандидатів для **3** питань («Скільки днів дають на виселення…», «Під яким кодом F2…», «Що розуміють під самоплагіатом?»). Вибірка мала (14 питань, які я підбирав сам), тож це ілюстрація механізмів, а не оцінка якості методів.

**BM25 вище, ніж вектор** (найбільша різниця): «Яка допомога студентові в академічній відпустці за медичними показаннями?» — позиція 1 проти 7. Усі три перші фрагменти векторного пошуку з правильного документа 340 (правила призначення стипендій), але це різні фрагменти про стипендії: тема збігається з усіма ними, і точний фрагмент із «50 відсотків мінімальної…» вектор поставив лише сьомим. BM25 ранжує за спільними словами питання, тому на перше місце вийшов фрагмент, що найбільше збігається з ним буквально. Так само BM25 виграв у питаннях про вибіркові курси (1 проти 4) і про стипендію ректора (1 проти 2).

**Вектор вище, ніж BM25:** «Що розуміють під самоплагіатом?» — позиція 1 проти «поза топ-20». У документі стоїть «Самоплагіат –» (називний відмінок), у питанні — «самоплагіатом» (орудний). Наш токенізатор (`\w+` у нижньому регістрі) не зводить слова до основи, тож для BM25 це різні слова, а спільні слова питання («що», «розуміють», «під») є в багатьох фрагментах, тому в топі BM25 лише загальні фрагменти («3.5. Академічна доброчесність під час оцінювання…», «Під час поточного контролю…»). Показова мінімальна пара: «Що таке самоплагіат?» — BM25 на 2-й позиції, «Що розуміють під самоплагіатом?» — поза топ-20, а вектор в обох випадках на 1-й. Так само питання «Чи виплачують академічну стипендію щомісяця?» (у документі «один раз на місяць») — вектор 1, BM25 9: спільних слів немає.

**Де прогноз не підтвердився.** Я очікував, що BM25 виграватиме на кодах «F2» і «FX». Для «FX» обидва методи поділили 4-те місце, а для «F2» BM25 узагалі не знайшов фрагмент серед 20 (вектор — 5-й). Чому саме так, з виводу не видно, я цього не досліджував; можливо, внесок рідкісного токена «f2» тоне серед загальних слів питання («під», «яким», «кодом», «спеціальність», «програмі»), які є в багатьох фрагментах. Висновок: помилки методів **різні** (вектор помиляється на точному формулюванні в межах однієї теми, BM25 — на формах слів і синонімах), і саме тому їх вигідно поєднувати (розділ 5).

### Питання 25. Обчисліть «вручну» бал RRF для фрагмента, що стоїть на 2-й позиції у векторному пошуку й на 5-й позиції в BM25 (розділ 5). Чому RRF використовує лише позиції, а не самі оцінки методів пошуку?

Фрагмент на позиції $r$ у списку отримує $\frac{1}{60 + r}$, бали з усіх списків додаються. Для позицій 2 і 5:

$$\mathrm{RRF} = \frac{1}{60+2} + \frac{1}{60+5} = \frac{1}{62} + \frac{1}{65} \approx 0{,}016129 + 0{,}015385 = 0{,}031514.$$

Нижче те саме обчислено кодом і перевірено на функції `rrf` із розділу 5: у два списки по 20 документів поставлено цільовий фрагмент (2-й у векторному і 5-й у BM25) і ще один (1-й і 3-й), бал якого мав би бути $\frac{1}{61}+\frac{1}{63}\approx 0{,}032266$, тобто вищий.

**Чому лише позиції.** Оцінки двох методів несумірні: косинусна подібність лежить у вузькому діапазоні (на практиці 0,3–0,8), а BM25 не обмежена зверху й залежить від довжини запиту та статистики корпусу (десятки одиниць). Звести їх до однієї шкали можна лише нормалізацією, яка чутлива до викидів і до набору кандидатів; проста сума сирих оцінок фактично віддала б усю вагу BM25. Позиції не залежать від шкали, їх легко порівнювати, а константа 60 згладжує вплив перших місць (різниця між 1-м і 2-м місцем невелика, тож один метод не може одноосібно «перекричати» інший). Мінус: RRF не бачить, наскільки сильно перший результат кращий за другий. Код нижче показує різницю шкал і порівнює три способи злиття на бенчмарку.

In [21]:
def rrf_score(ranks, c=60):
    return sum(1 / (c + rank) for rank in ranks)


manual = rrf_score([2, 5])
print(f"Вручну: 1/(60+2) + 1/(60+5) = {1 / 62:.6f} + {1 / 65:.6f} = {manual:.6f}")
other = rrf_score([1, 3])
print(f"Конкурент (1-й і 3-й): 1/61 + 1/63 = {other:.6f}")

# Перевірка функції rrf із розділу 5 на синтетичних списках
vector_list = [Document(page_content=f"вектор-{i}") for i in range(1, 21)]
bm25_list = [Document(page_content=f"bm25-{i}") for i in range(1, 21)]
target, rival = Document(page_content="ЦІЛЬОВИЙ"), Document(page_content="КОНКУРЕНТ")
vector_list[0], vector_list[1] = rival, target     # конкурент: 1-й у векторному, цільовий: 2-й
bm25_list[2], bm25_list[4] = rival, target         # конкурент: 3-й у BM25, цільовий: 5-й
fused = rrf([vector_list, bm25_list])
print("Порядок після rrf:", [d.page_content for d in fused[:4]])
assert fused[0] is rival and fused[1] is target, "порядок має збігатися з обчисленням вручну"
print("Порядок збігається з обчисленням вручну: конкурент (0.032266) > цільовий (0.031514) > решта (≤ 0.016393)")

# Різні шкали оцінок на справжніх запитах
print("\nДіапазон оцінок у 5 найкращих результатах:")
scale_rows = []
for question, _, _ in answerable[:6]:
    q_vec = np.array(embeddings.embed_query(question), dtype=np.float32)
    cosine_scores = np.sort(chunk_matrix @ q_vec)[::-1][:5]
    bm25_scores = np.sort(bm25.vectorizer.get_scores(tokenize(question)))[::-1][:5]
    scale_rows.append({"питання": question[:48], "косинус": f"{cosine_scores[-1]:.2f}–{cosine_scores[0]:.2f}", "BM25": f"{bm25_scores[-1]:.1f}–{bm25_scores[0]:.1f}"})
display(pd.DataFrame(scale_rows))


def make_fusion(mode):
    def fusion_search(query, k):
        q_vec = np.array(embeddings.embed_query(query), dtype=np.float32)
        cosine_scores = chunk_matrix @ q_vec
        bm25_scores = bm25.vectorizer.get_scores(tokenize(query))
        if mode == "сума сирих оцінок":
            fused_scores = cosine_scores + bm25_scores
        else:  # min-max нормалізація кожного списку, потім сума
            fused_scores = sum((s - s.min()) / (s.max() - s.min() + 1e-9) for s in (cosine_scores, bm25_scores))
        return [chunks[i] for i in np.argsort(-fused_scores)[:k]]
    return fusion_search


def metrics_on(search, items):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in items]
    return {"hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


fusion_rows = []
for name, search in [("лише вектор", vector_search), ("лише BM25", bm25_search), ("сума сирих оцінок", make_fusion("сума сирих оцінок")),
                     ("min-max + сума", make_fusion("min-max")), ("RRF (розділ 5)", hybrid_search)]:
    fusion_rows.append({"злиття": name, **metrics_on(search, answerable)})
display(pd.DataFrame(fusion_rows).round(2))

Вручну: 1/(60+2) + 1/(60+5) = 0.016129 + 0.015385 = 0.031514
Конкурент (1-й і 3-й): 1/61 + 1/63 = 0.032266
Порядок після rrf: ['КОНКУРЕНТ', 'ЦІЛЬОВИЙ', 'bm25-1', 'bm25-2']
Порядок збігається з обчисленням вручну: конкурент (0.032266) > цільовий (0.031514) > решта (≤ 0.016393)

Діапазон оцінок у 5 найкращих результатах:


,питання,косинус,BM25
0,До якої дати потрібно оплатити гуртожиток за пер,0.57–0.77,6.9–9.6
1,До якої дати оплачують гуртожиток за другий семе,0.53–0.71,6.9–10.3
2,З якої до якої години в гуртожитку має бути тиша,0.56–0.62,11.4–23.2
3,У який час гості можуть перебувати в гуртожитку?,0.65–0.72,11.3–16.2
4,Чи можна тримати в гуртожитку домашню тварину?,0.54–0.57,6.5–8.4
5,На який строк укладають договір найму житла в гу,0.56–0.64,11.3–18.3


,злиття,hit@1,hit@5,MRR
0,лише вектор,0.61,0.87,0.74
1,лише BM25,0.52,0.65,0.59
2,сума сирих оцінок,0.52,0.65,0.60
3,min-max + сума,0.57,0.96,0.72
4,RRF (розділ 5),0.61,0.91,0.76


**Коментар до результатів.** Обчислення вручну й функція `rrf` збігаються: цільовий фрагмент (2-й і 5-й) отримує 0,031514 і програє конкуренту (1-й і 3-й) із 0,032266, а фрагменти, що є лише в одному списку, мають не більше 0,016393 (1/61).

Шкали оцінок справді несумірні. У п'яти найкращих результатах косинусна подібність лежить у межах 0,53–0,77, а BM25 — 6,5–23,2: значення приблизно на порядок більші, і розкид у топ-5 теж ширший (до 11,8 одиниці проти не більше 0,2 у косинуса). Наслідок видно в таблиці: **проста сума сирих оцінок** дала hit@1 0,52, hit@5 0,65 і MRR 0,60, майже те саме, що «лише BM25» (0,52; 0,65; 0,59). BM25 «перекричав» косинус, і замість злиття вийшов BM25.

**Min-max + сума** має найвищий hit@5 (0,96), але нижчі hit@1 (0,57) і MRR (0,72), ніж у вектора (0,61; 0,74) і RRF (0,61; 0,76). RRF має найвищий MRR з п'яти способів (0,76), hit@1 на рівні вектора (0,61) і hit@5 0,91. Різниці між RRF, min-max і вектором — це одне-два питання з 23 (0,04 за hit@5 — рівно одне питання), тому «RRF найкращий» вони не доводять; переконливо видно лише, що сума сирих оцінок значно гірша.

### Питання 26. Чому cross-encoder (розділ 6) застосовують лише до 20 кандидатів, а не до всіх фрагментів корпусу? Скільки проходів моделі знадобилося б на одне питання, щоб переранжувати весь корпус?

Bi-encoder кодує фрагменти **заздалегідь і незалежно від запиту**: на питання потрібен один прохід моделі (для запиту) плюс множення матриці. Cross-encoder читає **пару** «запит + фрагмент» разом, тому оцінку не можна порахувати наперед: для кожного питання потрібен окремий прохід моделі на кожну пару. Для всього корпусу це $N$ проходів на питання, де $N$ — кількість фрагментів; для 20 кандидатів — 20. Кожен прохід — це велика (понад пів мільярда параметрів) модель на тексті близько тисячі символів, тож час відповіді й вартість зростають пропорційно $N$, а при зростанні корпусу це стає неприйнятним. Двоетапна схема (швидкий пошук → точне переранжування невеликої кількості кандидатів) використовує сильні сторони обох моделей. Ціна — що переранжування не може знайти те, чого немає серед кандидатів: якість обмежена **повнотою на 20 кандидатах**. Код нижче вимірює час і кількість проходів та порівнює переранжування кандидатів із переранжуванням усього корпусу (для чотирьох питань, тому клітинка виконується кілька хвилин).

In [22]:
def timed_predict(pairs):
    if device == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
    if device == "cuda":
        torch.cuda.synchronize()
    return np.asarray(scores), time.perf_counter() - started


n_chunks = len(chunks)
print(f"Фрагментів у корпусі: N = {n_chunks}. Проходів cross-encoder на одне питання: {CANDIDATES} (кандидати) проти {n_chunks} (весь корпус), у {n_chunks / CANDIDATES:.0f} раз більше.")

timed_predict([("прогрів", chunks[0].page_content)] * 2)  # прогрів
sample_items = answerable[::6]
rows = []
for question, gold, evidence in sample_items:
    candidates = hybrid_search(question, CANDIDATES)
    scores_candidates, seconds_candidates = timed_predict([(question, d.page_content) for d in candidates])
    scores_all, seconds_all = timed_predict([(question, d.page_content) for d in chunks])
    rank_candidates = first_hit_rank([candidates[i] for i in np.argsort(-scores_candidates)[:10]], gold, evidence)
    rank_all = first_hit_rank([chunks[i] for i in np.argsort(-scores_all)[:10]], gold, evidence)
    rows.append({"питання": question[:46], "с (20)": round(seconds_candidates, 2), "с (весь корпус)": round(seconds_all, 1),
                 "позиція (20 кандидатів)": rank_candidates, "позиція (весь корпус)": rank_all})
display(pd.DataFrame(rows))

candidate_ranks = [first_hit_rank(hybrid_search(question, CANDIDATES), gold, evidence) for question, gold, evidence in answerable]
print(f"Повнота гібридного пошуку на 20 кандидатах (верхня межа для переранжування): {np.mean([r is not None for r in candidate_ranks]):.0%} "
      f"({sum(r is not None for r in candidate_ranks)} з {len(answerable)} питань)")

Фрагментів у корпусі: N = 385. Проходів cross-encoder на одне питання: 20 (кандидати) проти 385 (весь корпус), у 19 раз більше.


,питання,с (20),с (весь корпус),позиція (20 кандидатів),позиція (весь корпус)
0,До якої дати потрібно оплатити гуртожиток за п,0.25,4.7,1,1
1,За який час потрібно звільнити гуртожиток післ,0.25,4.8,1,1
2,До якої дати викладачі вносять анотації вибірк,0.28,4.9,2,2
3,До якої дати проводять вибори студентського ом,0.25,4.9,1,1


Повнота гібридного пошуку на 20 кандидатах (верхня межа для переранжування): 100% (23 з 23 питань)


**Коментар до результатів.** У корпусі N = 385 фрагментів, тож переранжування всього корпусу — це 385 проходів cross-encoder на питання проти 20 для кандидатів, у **19 разів** більше. Час зростає так само: на чотирьох перевірених питаннях 20 кандидатів обробляються за **0,25–0,28 с**, увесь корпус — за **4,7–4,9 с**, тобто у 18–20 разів довше (на GPU, `device = cuda`).

Якість при цьому не змінилася: позиція правильного фрагмента для всіх чотирьох питань однакова — 1, 1, 2, 1 — і для 20 кандидатів, і для всього корпусу. Додаткові 365 пар коштували близько 4,6 с на питання й нічого не дали. Чотири питання — мало, але причину видно з останнього рядка виводу: повнота гібридного пошуку на 20 кандидатах дорівнює **100 %** (23 з 23), тож на цьому бенчмарку обмеження «переранжувати можна лише те, що знайдено» не заважає. На більшому корпусі чи складніших питаннях повнота на 20 кандидатах буде нижчою, і тоді кількість кандидатів стає параметром, який підбирають: кожен додатковий кандидат — це лінійне зростання часу. Для масштабу: ≈ 0,0125 с на пару дає для корпусу зі 100 000 фрагментів близько 21 хв на питання (це лінійна екстраполяція, а не вимір).

### Питання 27. Додайте до бенчмарку (розділ 7) три власні питання: одне з відповіддю, одне, що перефразовує наявне питання, і одне без відповіді в документах. Як змінилися метрики ретриву?

Додаю три питання: **нове з відповіддю** (оцінка «відмінно» A, діапазон 90–100 балів, у тому самому документі про оцінювання, що й питання про B і FX); **перефразування** питання про вибори омбудсмана («До якої дати проводять вибори…» → «До якого числа жовтня обирають…»); **без відповіді** (вартість проїзду маршруткою, якої в університетських положеннях немає). Нові питання записую в окремий список `BENCHMARK_PLUS`, щоб не змінювати початковий бенчмарк, на якому вже пораховано `evaluation`. Метрики ретриву рахуються лише на питаннях із відповіддю: питання без відповіді не має «правильного фрагмента», тому для нього натомість дивимося, наскільки високою є найкраща подібність — чи можна за нею відрізнити питання без відповіді.

In [23]:
NEW_QUESTIONS = [
    ("Скільки балів потрібно для оцінки «відмінно» A?", ["275"], r"90 ?[–-] ?100", "90–100 балів."),
    ("До якого числа жовтня обирають студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Скільки коштує проїзд маршруткою від вокзалу до університету?", [], None, None),
]
BENCHMARK_PLUS = BENCHMARK + NEW_QUESTIONS
answerable_plus = [(question, gold, evidence) for question, gold, evidence, _ in BENCHMARK_PLUS if gold]

for question, gold, evidence, _ in NEW_QUESTIONS:
    if gold and not any(c.metadata["doc_id"] in gold and re.search(evidence, c.page_content) for c in chunks):
        print(f"УВАГА: вираз-доказ «{evidence}» не знайдено в корпусі — відповідь на «{question}» слід перевірити в документі")

methods = [vector_search, bm25_search, hybrid_search, rerank_search]
rows = []
for search in methods:
    before = metrics_on(search, answerable)
    after = metrics_on(search, answerable_plus)
    rows.append({"метод": search.__name__, "питань": f"{len(answerable)} → {len(answerable_plus)}",
                 "hit@1": f"{before['hit@1']:.2f} → {after['hit@1']:.2f}", "hit@5": f"{before['hit@5']:.2f} → {after['hit@5']:.2f}",
                 "MRR": f"{before['MRR']:.2f} → {after['MRR']:.2f}"})
display(pd.DataFrame(rows))

print("Позиції правильного фрагмента для нових питань із відповіддю:")
rank_rows = []
for question, gold, evidence, _ in NEW_QUESTIONS:
    if gold:
        rank_rows.append({"питання": question, **{search.__name__: first_hit_rank(search(question, 10), gold, evidence) for search in methods}})
display(pd.DataFrame(rank_rows))

top_scores = {question: vector_store.similarity_search_with_score(question, k=1)[0][1] for question, *_ in BENCHMARK_PLUS}
answerable_scores = [top_scores[question] for question, gold, *_ in BENCHMARK_PLUS if gold]
unanswerable_scores = {question[:60]: round(float(top_scores[question]), 3) for question, gold, *_ in BENCHMARK_PLUS if not gold}
print(f"\nНайкраща косинусна подібність: для питань із відповіддю — від {min(answerable_scores):.2f} до {max(answerable_scores):.2f} "
      f"(медіана {np.median(answerable_scores):.2f}); для питань без відповіді: {unanswerable_scores}")

print("\nВідповіді системи на три нові питання:")
new_evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in NEW_QUESTIONS])
for _, row in new_evaluation.iterrows():
    print(f"- {row['питання']}\n  правильно: {row['правильно']}, цитування: {row['цитування']}, знайдено: {row['знайдено']}\n  відповідь: {row['відповідь']}")

,метод,питань,hit@1,hit@5,MRR
0,vector_search,23 → 25,0.61 → 0.60,0.87 → 0.88,0.74 → 0.73
1,bm25_search,23 → 25,0.52 → 0.52,0.65 → 0.68,0.59 → 0.59
2,hybrid_search,23 → 25,0.61 → 0.64,0.91 → 0.92,0.76 → 0.78
3,rerank_search,23 → 25,0.83 → 0.84,0.96 → 0.96,0.90 → 0.91


Позиції правильного фрагмента для нових питань із відповіддю:


,питання,vector_search,bm25_search,hybrid_search,rerank_search
0,Скільки балів потрібно для оцінки «відмінно» A?,5,4,1,1
1,До якого числа жовтня обирають студентського о...,1,1,1,1



Найкраща косинусна подібність: для питань із відповіддю — від 0.39 до 0.83 (медіана 0.66); для питань без відповіді: {'Скільки коштує обід у студентській їдальні?': 0.49, 'Який графік роботи бібліотеки університету?': 0.49, 'Скільки коштує рік навчання на контракті за спеціальністю «І': 0.706, 'Скільки коштує проїзд маршруткою від вокзалу до університету': 0.493}

Відповіді системи на три нові питання:
- Скільки балів потрібно для оцінки «відмінно» A?
  правильно: True, цитування: True, знайдено: ['275', '275-v3.1', '3320', '275', '275-v3.1']
  відповідь: 90 – 100 балів [1].
- До якого числа жовтня обирають студентського омбудсмана?
  правильно: True, цитування: True, знайдено: ['4389', '4389', '4389', '4389', '4389']
  відповідь: До 01 жовтня кожного навчального року обирають студентського омбудсмана [1].
- Скільки коштує проїзд маршруткою від вокзалу до університету?
  правильно: True, цитування: None, знайдено: ['348', '348', '291', '348', '348']
  відповідь: У наданих документах н

**Як змінилися метрики.** Додано три питання: «відмінно» A (відповідь `90 – 100 балів`), перефразування питання про омбудсмана й питання без відповіді (маршрутка). До метрик ретриву потрапили два питання з відповіддю, тож вибірка зросла з 23 до 25:

| метод | hit@1 | hit@5 | MRR |
|---|---|---|---|
| вектор | 0,61 → 0,60 | 0,87 → 0,88 | 0,74 → 0,73 |
| BM25 | 0,52 → 0,52 | 0,65 → 0,68 | 0,59 → 0,59 |
| гібрид (RRF) | 0,61 → 0,64 | 0,91 → 0,92 | 0,76 → 0,78 |
| з переранжуванням | 0,83 → 0,84 | 0,96 → 0,96 | 0,90 → 0,91 |

Зміни не перевищують 0,03: нове питання важить лише 4 п.п. вибірки, а різницю між методами цей зсув не стирає (порядок методів не змінився). Позиції нових питань пояснюють, чому гібрид трохи виграв: питання про «відмінно» A вектор поставив 5-м, BM25 — 4-м, а гібрид і переранжування — 1-м (фрагмент стояв високо в обох списках, і RRF додав обидва внески). Перефразоване питання знайдено на 1-й позиції **усіма чотирьома** методами. Це м'яке перефразування: нове питання зберегло слова «студентського омбудсмана» і навіть додало «жовтня», яке є у фрагменті («до 01 жовтня»), тож BM25 його не розгубив; для перевірки на синонімах потрібне жорсткіше перефразування.

**Питання без відповіді.** Найкраща косинусна подібність для чотирьох питань без відповіді: 0,49 (їдальня), 0,49 (бібліотека), 0,493 (маршрутка) і **0,706** (вартість навчання на контракті); для питань із відповіддю — від 0,39 до 0,83 (медіана 0,66). Отже, поріг подібності відмови не розділяє: поріг вище 0,706 відсікав би щонайменше половину питань із відповіддю, а навіть поріг 0,5 відсікає принаймні одне питання з відповіддю (мінімум 0,39) і водночас пропускає питання про контракт. Тому відмову має ухвалювати генератор за змістом фрагментів, і на всіх трьох нових питаннях система відповіла правильно: два питання з відповіддю — правильна відповідь із правильним джерелом, питання про маршрутку — «У наданих документах немає відповіді.»

### Питання 28. Порівняйте метрики ретриву для чотирьох методів пошуку (розділ 7). Чому переранжування найсильніше впливає саме на hit@1?

In [24]:
import math


def wilson(hits_count, total, z=1.96):
    p = hits_count / total
    denom = 1 + z * z / total
    centre = (p + z * z / (2 * total)) / denom
    half = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


MAX_K = 20
all_ranks = {search.__name__: [first_hit_rank(search(question, MAX_K), gold, evidence) for question, gold, evidence in answerable] for search in methods}


def hit_at(ranks, k):
    return sum(r is not None and r <= k for r in ranks)


rows = []
for name, ranks in all_ranks.items():
    low, high = wilson(hit_at(ranks, 1), len(ranks))
    rows.append({"метод": name, "hit@1": hit_at(ranks, 1) / len(ranks), "ДІ hit@1": f"[{low:.2f}; {high:.2f}]", "hit@3": hit_at(ranks, 3) / len(ranks),
                 "hit@5": hit_at(ranks, 5) / len(ranks), "hit@10": hit_at(ranks, 10) / len(ranks), "hit@20": hit_at(ranks, 20) / len(ranks),
                 "MRR@10": np.mean([1 / r if r and r <= 10 else 0 for r in ranks])})
display(pd.DataFrame(rows).round(2))

print("Де стоїть правильний фрагмент (кількість питань):")
buckets = [("1", lambda r: r == 1), ("2–5", lambda r: r is not None and 2 <= r <= 5), ("6–10", lambda r: r is not None and 6 <= r <= 10),
           ("11–20", lambda r: r is not None and r > 10), ("немає", lambda r: r is None)]
display(pd.DataFrame({name: {label: sum(test(r) for r in ranks) for label, test in buckets} for name, ranks in all_ranks.items()}))

hybrid_ranks, rerank_ranks = all_ranks["hybrid_search"], all_ranks["rerank_search"]
up = sum((b or MISSING) < (a or MISSING) for a, b in zip(hybrid_ranks, rerank_ranks))
down = sum((b or MISSING) > (a or MISSING) for a, b in zip(hybrid_ranks, rerank_ranks))
print(f"\nПереранжування порівняно з гібридним пошуком: піднялося {up}, опустилося {down}, без змін {len(hybrid_ranks) - up - down} питань")
print(f"hit@20: гібридний {hit_at(hybrid_ranks, 20)}, після переранжування {hit_at(rerank_ranks, 20)} (множина кандидатів та сама)")

,метод,hit@1,ДІ hit@1,hit@3,hit@5,hit@10,hit@20,MRR@10
0,vector_search,0.61,[0.41; 0.78],0.83,0.87,1.00,1.00,0.74
1,bm25_search,0.52,[0.33; 0.71],0.57,0.65,0.87,0.96,0.59
2,hybrid_search,0.61,[0.41; 0.78],0.91,0.91,1.00,1.00,0.76
3,rerank_search,0.83,[0.63; 0.93],0.96,0.96,1.00,1.00,0.90


Де стоїть правильний фрагмент (кількість питань):


,vector_search,bm25_search,hybrid_search,rerank_search
1,14,12,14,19
2–5,6,3,7,3
6–10,3,5,2,1
11–20,0,2,0,0
немає,0,1,0,0



Переранжування порівняно з гібридним пошуком: піднялося 8, опустилося 2, без змін 13 питань
hit@20: гібридний 23, після переранжування 23 (множина кандидатів та сама)


**Порівняння методів.** hit@1 / hit@5 / MRR@10 на 23 питаннях: вектор — 0,61 / 0,87 / 0,74; BM25 — 0,52 / 0,65 / 0,59; гібрид (RRF) — 0,61 / 0,91 / 0,76; з переранжуванням — **0,83 / 0,96 / 0,90**.

- **BM25 найслабший:** 3 питання поза топ-10 (два на позиціях 11–20, одне поза топ-20; hit@20 = 0,96).
- **Вектор і гібрид** мають однаковий hit@1 (14 із 23), але гібрид підтягує правильний фрагмент із 6–10-ї позиції у 2–5-ту: hit@3 зріс з 0,83 до 0,91, hit@5 — з 0,87 до 0,91.
- **Переранжування:** правильний фрагмент на 1-му місці у 19 із 23 питань (гібрид — 14). Відносно гібриду піднялося 8 питань, опустилося 2, без змін 13. Напрямок узгоджений, але для 10 змін це ще не доказ: двосторонній тест знаків дає p ≈ 0,11.
- **Довірчі інтервали** hit@1 за Вілсоном перекриваються для всіх пар, навіть для BM25 ([0,33; 0,71]) і переранжування ([0,63; 0,93]), тож порядок методів на 23 питаннях — орієнтир, а не статистично доведений висновок.

**Чому переранжування найсильніше впливає на hit@1.** Cross-encoder отримує лише ті самі 20 кандидатів, які знайшов гібридний пошук, і лише **змінює їхній порядок**: він не може знайти фрагмент, якого серед кандидатів немає (тому hit@20 не змінюється, що підтверджує останній рядок виводу). Значущою є лише зміна порядку на початку списку. Швидкі методи (bi-encoder, BM25) добре знаходять тематично близькі фрагменти, і правильний фрагмент зазвичай уже в топ-5, але серед кількох схожих за темою фрагментів їм важко визначити той, що справді містить відповідь; cross-encoder читає запит і фрагмент разом і розрізняє «стосується теми» та «відповідає на питання». hit@1 — найсуворіша метрика порядку (правильний мусить бути першим), тож резерв для покращення тут найбільший, а hit@5 і hit@10 уже близькі до стелі (повноти кандидатів), тому там переранжування мало що змінює. MRR реагує проміжно: зростання позицій з 2–5 на 1 піднімає його, але лише частково. На 23 питаннях висновок обережний: одне питання — понад 4 відсоткові пункти, а довірчі інтервали Вілсона перекриваються.

### Питання 29. У розділі 9 суддею виступає та сама модель, що генерує відповіді. Які проблеми це може спричинити? Перевірте вручну вердикти судді для пʼяти відповідей і порівняйте зі своїми.

Проблеми, коли суддя і генератор — одна модель:

1. **Упередженість до власних відповідей** (self-preference): LLM-судді схильні вищо оцінювати текст, написаний у їхньому стилі й за їхнім розподілом ймовірностей.
2. **Спільні сліпі зони**: якщо модель неправильно зрозуміла фрагмент чи питання, вона так само неправильно зрозуміє й порівняння з еталоном; помилки генератора і судді **корелюють**, тож суддя не виявить саме ті помилки, які й треба знайти.
3. **Суддя теж галюцинує і чутливий до формулювання**: довга, впевнена, добре структурована відповідь може здаватися правильною, а коротка правильна — ні; на результат впливають порядок і формулювання промпту судді.
4. **Невеликі моделі слабші за ті, що оцінюють**: 4-мільярдна модель — слабкий суддя.
5. **Відтворюваність і приховані зсуви**: зміна судді чи його промпту змінює метрики, тож порівнювати системи можна лише з тим самим суддею.

Способи зменшити ризик: інша (сильніша) модель-суддя, правила там, де їх можна застосувати (збіг чисел, дат, ключових слів), вибіркова перевірка вручну, контрольні приклади. Код нижче: (а) **позитивний контроль** — суддя бачить еталон як відповідь і має сказати «так»; (б) **негативний контроль** — суддя бачить еталонну відповідь на *інше* питання і має сказати «ні»; (в) порівняння вердиктів судді з простим правилом (усі числа з еталона є у відповіді); (г) відбір п'яти відповідей для ручної перевірки.

In [25]:
references = [(question, reference) for question, gold, _, reference in BENCHMARK if gold]
positive = [judge(question, reference, reference) for question, reference in references]
negative = [judge(question, reference, references[(i + 1) % len(references)][1]) for i, (question, reference) in enumerate(references)]
print(f"Позитивний контроль (еталон як відповідь): суддя сказав «так» у {sum(positive)} з {len(positive)} випадків (мало бути {len(positive)})")
print(f"Негативний контроль (еталон іншого питання): суддя сказав «так» у {sum(negative)} з {len(negative)} випадків (мало бути 0)")
for i, ((question, _), verdict) in enumerate(zip(references, negative)):
    if verdict:
        print(f"   хибне «так»: «{question}» — відповідь «{references[(i + 1) % len(references)][1]}»")


def key_numbers(text):
    return set(re.findall(r"\d+", text or ""))


def rule_verdict(reference, answer):
    reference_numbers = key_numbers(reference)
    if not reference_numbers:
        return None  # правило не застосовується до відповідей без чисел
    return reference_numbers <= key_numbers(answer)


bench_rows = evaluation[has_answer].copy()
bench_rows["еталон"] = [reference for _, gold, _, reference in BENCHMARK if gold]
bench_rows["правило"] = [rule_verdict(r, a) for r, a in zip(bench_rows["еталон"], bench_rows["відповідь"])]
applicable = bench_rows[bench_rows["правило"].notna()]
print("\nВердикт судді проти простого правила «усі числа з еталона є у відповіді» (лише питання з числами в еталоні):")
display(pd.crosstab(applicable["правильно"].map({True: "суддя: так", False: "суддя: ні"}), applicable["правило"].astype(bool).map({True: "правило: так", False: "правило: ні"})))

# П'ять відповідей для ручної перевірки: спершу ті, де суддя і правило розходяться, потім випадкові з решти
disagree = applicable[applicable["правильно"] != applicable["правило"].astype(bool)]
rest = bench_rows.drop(index=disagree.index)
manual_rows = pd.concat([disagree.head(3), rest.sample(n=5 - min(3, len(disagree)), random_state=0)])
print(f"\nДля ручної перевірки відібрано {len(manual_rows)} відповідей (розбіжностей суддя/правило: {len(disagree)}):\n")
for number, (_, row) in enumerate(manual_rows.iterrows(), start=1):
    print(f"{number}. Питання: {row['питання']}\n   Еталон: {row['еталон']}\n   Відповідь системи: {row['відповідь']}\n   Вердикт судді: {'так' if row['правильно'] else 'ні'}\n")

Позитивний контроль (еталон як відповідь): суддя сказав «так» у 23 з 23 випадків (мало бути 23)
Негативний контроль (еталон іншого питання): суддя сказав «так» у 0 з 23 випадків (мало бути 0)

Вердикт судді проти простого правила «усі числа з еталона є у відповіді» (лише питання з числами в еталоні):


правило,правило: ні,правило: так
правильно,,
суддя: ні,0,1
суддя: так,1,11



Для ручної перевірки відібрано 5 відповідей (розбіжностей суддя/правило: 2):

1. Питання: У який час гості можуть перебувати в гуртожитку?
   Еталон: З 8:00 до 22:00.
   Відповідь системи: Гості можуть перебувати в гуртожитку з 08:00 до 22:00 години [1].
   Вердикт судді: так

2. Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
   Еталон: F2.
   Відповідь системи: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
   Вердикт судді: ні

3. Питання: Скільки балів потрібно для оцінки «добре» B?
   Еталон: 82–89 балів.
   Відповідь системи: Для оцінки «добре» B потрібно 82 – 89 балів [2].
   Вердикт судді: так

4. Питання: За яким показником формують рейтинг для призначення академічної стипендії?
   Еталон: За середнім балом семестрового контролю.
   Відповідь системи: Рейтинг для призначення академічної стипендії формують за середнім балом семестрового контролю [1].
   Вердикт судді: так

5. Питан

Ручна перевірка. Клітинку нижче запущено з порожнім словником `MANUAL_VERDICTS` (її запустили до того, як я переглянув відібрані відповіді), тому вона друкує «Ручні вердикти ще не записано». Щоб не перезапускати весь ноутбук заради одного словника, мої вердикти й порівняння з суддею наведено в таблиці одразу після клітинки.

In [26]:
MANUAL_VERDICTS = {}  # {текст питання: True/False}; заповнюється після перегляду відібраних вище відповідей

if MANUAL_VERDICTS:
    comparison_rows = []
    for _, row in manual_rows.iterrows():
        mine = MANUAL_VERDICTS.get(row["питання"])
        comparison_rows.append({"питання": row["питання"][:60], "суддя": row["правильно"], "вручну": mine, "збіг": row["правильно"] == mine})
    comparison_table = pd.DataFrame(comparison_rows)
    display(comparison_table)
    print(f"Вердикти судді збіглися з ручними у {int(comparison_table['збіг'].sum())} з {len(comparison_table)} випадків")
else:
    print("Ручні вердикти ще не записано.")

Ручні вердикти ще не записано.


**Мої ручні вердикти** для п'яти відібраних відповідей (`True` — відповідь правильна й узгоджується з еталоном):

| № | Питання | Відповідь системи (еталон) | Суддя | Вручну | Збіг |
|---|---|---|---|---|---|
| 1 | У який час гості можуть перебувати в гуртожитку? | 08:00–22:00 [1] (8:00–22:00) | так | **True** | ✓ |
| 2 | Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі? | 121 [2] (F2) | ні | **False** | ✓ |
| 3 | Скільки балів потрібно для оцінки «добре» B? | 82 – 89 балів [2] (82–89) | так | **True** | ✓ |
| 4 | За яким показником формують рейтинг для призначення академічної стипендії? | за середнім балом семестрового контролю [1] (так само) | так | **True** | ✓ |
| 5 | Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС? | 240 кредитів ЄКТС [1][3][4] (240) | так | **True** | ✓ |

Судді збіглися з ручними у **5 із 5**. У № 1 «08:00» і «8:00» — той самий час (на цьому «посипалося» правило з чисел). У № 2 відповідь «121» суперечить еталону F2: хибно.

Щоб отримати таку таблицю кодом, достатньо заповнити словник у клітинці вище й запустити її знову:

```python
MANUAL_VERDICTS = {
    "У який час гості можуть перебувати в гуртожитку?": True,
    "Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?": False,
    "Скільки балів потрібно для оцінки «добре» B?": True,
    "За яким показником формують рейтинг для призначення академічної стипендії?": True,
    "Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?": True,
}
```

**Висновок про суддю.**

- **Контролі:** позитивний — 23 «так» із 23, негативний — 0 хибних «так» із 23: суддя не відповідає «так» на все й не відповідає «ні» на все. Але контролі перевіряють лише найгрубіші випадки (точна копія еталона й еталон іншого питання), тонких помилок вони не знаходять: цей самий суддя помилився на правильній відповіді з додатковою деталлю (питання 30).
- **Суддя проти правила** «усі числа з еталона є у відповіді» (13 питань із числами в еталоні): збіглися у **11** із 13 (усі — «так/так», випадків «ні/ні» немає), розбіжностей **2**. Обидві розбіжності — помилки *правила*, а не судді: (1) «08:00» проти «8:00» — правило порівнює рядки цифр, і «08» ≠ «8», тому правильна відповідь отримала «ні»; (2) у відповіді «121 [2]» правило знайшло цифру «2» з еталона «F2» у номері джерела `[2]` і визнало хибну відповідь правильною. Тож навіть «просте правило» не надійніше за суддю й саме потребує перевірки.
- **Ручна перевірка:** вердикти судді збіглися з моїми у **5 із 5** випадків.
- **Обмеження.** Вибірка мала (5 відповідей), і дві з них відібрано саме через розбіжність судді з правилом. Такий відбір знаходить помилки судді лише там, де правило застосовне, тобто де в еталоні є числа. Єдина реальна помилка судді в цьому запуску (питання про стипендію ректора, еталон без чисел, див. питання 30) у вибірку не потрапила, тож 5 із 5 не означає, що суддя безпомилковий. Упередженість до власних відповідей (self-preference) тут узагалі **не виміряна**: суддя — та сама модель, що генерує, а контролю з іншим суддею чи людською розміткою немає.
- **Практично:** підсумкові 91 % (21 з 23) — орієнтир. Якщо виправити єдине хибне «ні» вручну, виходить 22 з 23 (96 %).

### Питання 30. Розгляньте неправильні відповіді в розділі 10 «Аналіз відмов». Для кожної визначте тип помилки: промах ретриву, помилка генерації, помилка судді чи неоднозначний еталон. Обґрунтуйте.

Щоб класифікація спиралася на факти, а не на враження, для кожної неправильної відповіді збираю «справу»: чи є вираз-доказ серед п'яти фрагментів, які отримав генератор (`rerank_search(питання, 5)`), на які фрагменти посилається відповідь, чи містить відповідь ключові числа з еталона. Попередній тип за цими ознаками обчислюється правилами нижче; остаточний тип і обґрунтування записано після перегляду відповіді:

- **промах ретриву** — виразу-доказу немає серед фрагментів у промпті;
- **помилка генерації** — фрагмент із доказом у промпті, але відповідь неправильна (відмова, вигадка, змішування джерел) або система відповіла замість відмови;
- **помилка судді** — відповідь за змістом правильна (є ключові факти), але суддя сказав «ні» (чи навпаки);
- **неоднозначний еталон** — відповідь не суперечить документам, але спирається на інше місце чи документ, ніж еталон.

In [27]:
bench_by_question = {question: (gold, evidence, reference) for question, gold, evidence, reference in BENCHMARK}


def preliminary_type(row, in_context_rank, answer_has_facts):
    gold, _, _ = bench_by_question[row["питання"]]
    refused = REFUSAL in row["відповідь"].lower()
    if not gold:
        return "помилка генерації: відповідь замість відмови" if not refused else "перевірити вручну"
    if in_context_rank is None:
        return "промах ретриву (доказу немає у промпті)"
    if refused:
        return "помилка генерації: відмова попри наявний доказ"
    if answer_has_facts and not row["правильно"]:
        return "імовірно помилка судді (ключові факти у відповіді є)"
    if answer_has_facts and row["цитування"] in (False,):
        return "імовірно неоднозначний еталон або неточні номери джерел"
    if not answer_has_facts:
        return "помилка генерації (доказ у промпті, але відповідь не містить ключових фактів)"
    return "перевірити вручну"


case_rows = []
for _, row in failures.iterrows():
    gold, evidence, reference = bench_by_question[row["питання"]]
    prompt_docs = rerank_search(row["питання"], 5)
    in_context_rank = first_hit_rank(prompt_docs, gold, evidence) if gold else None
    reference_numbers = key_numbers(reference)
    answer_has_facts = (reference_numbers <= key_numbers(row["відповідь"])) if reference_numbers else None
    cited = sorted(set(cited_doc_ids(row["відповідь"], row["знайдено"])))
    kind = preliminary_type(row, in_context_rank, answer_has_facts)
    case_rows.append({"питання": row["питання"], "тип (попередньо)": kind})
    print(f"Питання: {row['питання']}")
    print(f"  еталон: {reference}")
    print(f"  відповідь: {row['відповідь']}")
    print(f"  правильний документ: {gold}; у промпті: {row['знайдено']}; доказ у промпті на позиції: {in_context_rank}; відповідь цитує: {cited}")
    print(f"  суддя: {'так' if row['правильно'] else 'ні'}; цитування: {row['цитування']}; ключові числа еталона у відповіді: {answer_has_facts}")
    print(f"  → попередній тип: {kind}\n")
print(f"Неправильних відповідей: {len(case_rows)} із {len(evaluation)}")

Питання: Для кого встановлюється стипендія ректора?
  еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
  правильний документ: ['344']; у промпті: ['344', '340', '344', '340', '340']; доказ у промпті на позиції: 1; відповідь цитує: ['344']
  суддя: ні; цитування: True; ключові числа еталона у відповіді: None
  → попередній тип: помилка генерації (доказ у промпті, але відповідь не містить ключових фактів)

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  еталон: F2.
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
  правильний документ: ['3320']; у промпті: ['3329', '3329', '3320', '3329', '33

**Остаточна класифікація.** У бенчмарку з 26 питань неправильних відповідей **дві**: правильність на питаннях із відповіддю — 21 з 23 (91 %), усі 3 відмови правильні.

| № | Питання | Попередній тип (за правилами) | Остаточний тип |
|---|---|---|---|
| 1 | Для кого встановлюється стипендія ректора? | помилка генерації | **помилка судді** |
| 2 | Який шифр спеціальності «Інженерія програмного забезпечення» в чинній програмі? | промах ретриву | **промах ретриву** (застаріла редакція документа; суперечливі джерела) |

**1. Стипендія ректора — помилка судді.** Ретрив у порядку: правильний документ 344 стоїть у промпті двічі, вираз-доказ — у фрагменті на 1-й позиції, а відповідь цитує саме `[1]` (цитування правильне). Відповідь містить усі елементи еталона: денну форму навчання та суттєві досягнення у навчанні, науковій, творчій і громадській діяльності. Відрізняється лише доданим «є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань», якого в еталоні немає. Промпт судді вимагає «ту саму інформацію, що й еталонна», тому, найімовірніше, суддя відхилив відповідь через додаткову частину (суддя повертає одне слово, тож причини ми не бачимо). Попереднє правило помилилося: в еталоні немає чисел, `answer_has_facts` став `None`, і умова «відповідь не містить ключових фактів» спрацювала хибно; правило не застосовне до еталонів без чисел. **Застереження:** додаткову частину я не звіряв із текстом фрагмента 344, у виводі його немає. Якщо її в документі немає, це вигадка моделі, і тип змінився б на помилку генерації. **Виправлення:** переформулювати промпт судді («чи містить відповідь усю інформацію з еталона й не суперечить їй?», тобто додаткові відомості допустимі) і перевіряти додаткові твердження окремою перевіркою на вірність фрагментам.

**2. Шифр спеціальності — промах ретриву.** Правильний документ 3320 (ОПП F2, редакція 2025) є в промпті лише на 3-й позиції з п'яти, а чотири фрагменти з п'яти взято з 3329 (ОПП 121, редакція 2024, застаріла). Виразу-доказу «спеціальність F2» серед п'яти фрагментів немає, відповідь цитує `[2]`, тобто документ 3329, тому цитування хибне. Модель вірно переказала те, що бачила (у застарілій редакції шифр 121), тож це не галюцинація; суддя справедливо сказав «ні», і моя ручна перевірка (№ 2 вище) з ним збіглася. Еталон не двозначний: питання питає про *чинну* програму, а F2 стоїть у назві програми 2025 року. **Виправлення:** позначати редакції в метаданих і відфільтровувати застарілі (як у демо 2). Але демо 2 показує, що самого фільтра мало: після нього всі п'ять фрагментів уже з чинної програми 3320, а модель усе одно відповіла «121», бо в самій чинній програмі 121 трапляється у 5 з 40 фрагментів, а F2 — у 2 з 40. Остаточно це виправляє власник документа; з боку RAG — показувати джерела з редакцією й повідомляти про суперечність.

**Підсумок.** Одна з двох помилок — на боці оцінювання (суддя), одна — на боці системи (ретрив і застарілі документи); помилок генерації й неоднозначних еталонів серед двох випадків немає. Двох випадків замало, щоб говорити про розподіл типів, вони лише ілюструють класифікацію.